# Medical Triage — GRPO Fine-tuning

Fine-tunes **Qwen2.5-1.5B-Instruct** using **Group Relative Policy Optimization (GRPO)**  
on the [Medical Triage Environment](https://huggingface.co/spaces/kunalkachru23/medical-triage-env).

The live HF Space acts as the reward oracle — no local server needed.

**Before running:** Runtime → Change runtime type → **T4 GPU**

## Quick Start (Portable Run Guide)

This notebook is designed to run in both **Google Colab** and **local Jupyter**.

### 1) Pick storage mode in Cell 1b
- Set `USE_GOOGLE_DRIVE = True` for Colab persistence across disconnects.
- Leave it `False` for local filesystem output.
- Artifacts/checkpoints are written under `DRIVE_DIR` (name configurable via `RUN_DIR_NAME`).

### 2) Set runtime config in Cell 3
- `SERVER_URL`: your deployed environment URL (or local API URL).
- `MODEL_ID`: base model to fine-tune.
- `HF_TOKEN`: optional; only needed when pushing adapter to Hugging Face Hub.

### 3) Fresh vs resume training in Cell 10
- `RUN_MODE = 'fresh'` for first run.
- `RUN_MODE = 'resume_latest'` to continue after interruption.
- `RUN_MODE = 'resume_path'` with `RESUME_CHECKPOINT_PATH` for manual checkpoint selection.

### 4) Training evidence outputs
- Reward telemetry JSONL: `grpo_reward_events.jsonl` under `OUTPUT_DIR`.
- Auto plots and summary render at end of training.
- Recovery/report cells (Cell 13-15) can rebuild visuals from telemetry or checkpoint `trainer_state.json` for historical runs.

In [ ]:
# Cell 1 — Install dependencies
!pip install trl>=0.12.0 transformers>=4.40.0 accelerate bitsandbytes peft datasets requests matplotlib -q

In [ ]:
# Cell 1b — Output directory setup (portable: works with or without Google Drive)
import os

# Set True when running in Colab and you want persistent checkpoints/artifacts on Drive.
USE_GOOGLE_DRIVE = False
RUN_DIR_NAME = 'grpo-medical-triage'

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_OUTPUT_ROOT = '/content/drive/MyDrive'
    print('Google Drive mounted.')
else:
    # Generic default: local runtime/workspace folder.
    BASE_OUTPUT_ROOT = os.getcwd()
    print('Using local filesystem output root (no Drive mount).')

DRIVE_DIR = os.path.join(BASE_OUTPUT_ROOT, RUN_DIR_NAME)
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Checkpoints and artifacts will be stored in: {DRIVE_DIR}')

In [ ]:
# Cell 2 — Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Cell 3 — Config (edit these values for your environment)
import os

# Use your deployed environment URL (or local server URL like http://localhost:8000).
SERVER_URL        = os.environ.get('SERVER_URL', 'https://kunalkachru23-medical-triage-env.hf.space')
MODEL_ID          = os.environ.get('MODEL_ID', 'Qwen/Qwen2.5-1.5B-Instruct')
HF_TOKEN          = os.environ.get('HF_TOKEN', '')  # optional — required only for push_to_hub

# OUTPUT_DIR comes from Cell 1b and is portable across Colab/local runs.
OUTPUT_DIR        = DRIVE_DIR
PROMPTS_PER_TASK  = 8   # 8 x 11 tasks = 88 prompts total
NUM_GENERATIONS   = 4   # GRPO group size G
EPOCHS            = 2   # 2 epochs on 88 prompts = ~176 optimizer steps

TASKS = [
    'simple_triage', 'conflicting_vitals', 'masked_deterioration',
    'demographic_fairness', 'deteriorating_patient', 'sepsis_bundle',
    'paediatric_triage', 'medication_reconciliation',
    'icu_deterioration', 'sbar_handover', 'differential_diagnosis',
]

print(f'SERVER_URL={SERVER_URL}')
print(f'OUTPUT_DIR={OUTPUT_DIR}')

In [ ]:
# Cell 4 — Health check
import requests
r = requests.get(f'{SERVER_URL}/health', timeout=15)
print('Server health:', r.json())

In [ ]:
# Cell 5 — Build dataset from live environment
from datasets import Dataset

records = []
for task in TASKS:
    count = 0
    for _ in range(PROMPTS_PER_TASK):
        try:
            r = requests.post(f'{SERVER_URL}/reset', json={'task_id': task}, timeout=30)
            r.raise_for_status()
            obs = r.json()['observation']
            patient_text = obs.get('patient_history') or obs.get('patient_presentation', '')
            # task_description removed — it uses different priority values and conflicts with system prompt
            prompt = f'[TASK: {task}]\n\n{patient_text}'.strip()
            records.append({'prompt': prompt, 'task': task})
            count += 1
        except Exception as e:
            print(f'Warning: /reset failed ({task}): {e}')
    print(f'  {task}: {count}/{PROMPTS_PER_TASK} prompts')

dataset = Dataset.from_list(records)
print(f'\nTotal: {len(dataset)} prompts across {len(TASKS)} tasks')
print('Sample:', dataset[0]['prompt'][:200])

In [ ]:
# Cell 6 — Load tokenizer + apply chat template
from transformers import AutoTokenizer

# NOTE:
# Keep labels and enum values aligned to server-side graders so GRPO receives
# clean reward signal (schema drift here directly lowers rewards).
SYSTEM_PROMPT = (
    'You are a clinical triage AI assistant.\n'
    'The user message begins with [TASK: <name>]. Respond with ONE single flat JSON object for that task only.\n'
    'No nested objects. No markdown. No code fences. No extra text.\n'
    'Always include every required key for that task; if uncertain, emit best-guess placeholder values instead of omitting keys.\n\n'
    'If [TASK: simple_triage] or [TASK: conflicting_vitals] or [TASK: masked_deterioration] or [TASK: demographic_fairness]:\n'
    '{"priority":"low|medium|high|critical","news2_score":<integer 0-20>,'
    '"critical_sign":"<most abnormal vital sign>","recommended_action":"emergency_response|urgent_review|routine_monitoring",'
    '"rationale":"<str>","confidence":<0.0-1.0>}\n\n'
    'If [TASK: deteriorating_patient]:\n'
    '{"action":"monitor|escalate|emergency_response","rationale":"<str>","confidence":<0.0-1.0>}\n\n'
    'If [TASK: sepsis_bundle]:\n'
    '{"priority":"low|medium|high|critical",'
    '"bundle_elements":["blood_cultures","broad_spectrum_antibiotics","iv_fluid_bolus","lactate_measurement","vasopressors"],'
    '"antibiotic_choice":"<str>","fluid_volume_ml":<integer>,"vasopressor_indicated":<true|false>,'
    '"rationale":"<str>","confidence":<0.0-1.0>}\n\n'
    'If [TASK: paediatric_triage]:\n'
    '{"priority":"low|medium|high|critical",'
    '"age_group":"infant|toddler|preschool|school_age|adolescent","pews_score":<integer 0-13>,'
    '"critical_sign":"<most abnormal vital sign>","recommended_action":"emergency_response|urgent_review|routine_monitoring",'
    '"rationale":"<str>","confidence":<0.0-1.0>}\n\n'
    'If [TASK: medication_reconciliation]:\n'
    '{"issues_found":["<str>"],"severity":"critical|high|medium|low","requires_pharmacist":<true|false>,'
    '"recommended_action":"safe_to_prescribe|modify_dose|withhold_drug|emergency_review",'
    '"drug_to_withhold":"<drug name or none>","rationale":"<str>","confidence":<0.0-1.0>}\n\n'
    'If [TASK: icu_deterioration]:\n'
    '{"sofa_score":<integer 0-24>,"primary_organ_failure":"cardiovascular|respiratory|renal|hepatic|neurological|coagulation",'
    '"deterioration_trend":"improving|stable|worsening",'
    '"intervention":"maintain_current|increase_support|emergency_escalation|prepare_palliation",'
    '"rationale":"<str>"}\n\n'
    'If [TASK: sbar_handover]:\n'
    '{"escalation_required":<true|false>,"priority":"low|medium|high|critical",'
    '"assessment":"<string describing clinical situation>",'
    '"recommendation":"routine_monitoring|urgent_review|emergency_response"}\n\n'
    'If [TASK: differential_diagnosis]:\n'
    '{"must_not_miss":"<life-threatening diagnosis to exclude>","top_diagnosis":"<most likely diagnosis>",'
    '"differentials":["<str>","<str>","<str>"],"first_investigation":"<most important first test>",'
    '"urgency":"immediate|urgent|routine"}'
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

def format_prompt(example):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': example['prompt']},
    ]
    example['prompt'] = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return example

dataset = dataset.map(format_prompt)
print('Tokenizer loaded.')
print('Sample (first 300 chars):', dataset[0]['prompt'][:300])

In [ ]:
# Cell 7 — Load model with 4-bit quantization
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

used_gb  = torch.cuda.memory_allocated() / 1e9
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'Model loaded. VRAM: {used_gb:.1f} GB used / {total_gb:.0f} GB total')

In [ ]:
# Cell 8 — Apply LoRA adapter
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 9 — Reward function (robust JSON extraction + schema token normalization + auto logging)
import os, uuid, time, re, json as _json
from requests.adapters import HTTPAdapter, Retry

REQUEST_TIMEOUT = 25
FALLBACK_REWARD = 0.0001
DEBUG_LOGGING = True

PRIORITY_MAP = {
    'immediate': 'critical',
    'urgent': 'high',
    'standard': 'medium',
    'non_urgent': 'low',
}
SEPSIS_BUNDLE_TOKEN_MAP = {
    'iv_antibiotics': 'broad_spectrum_antibiotics',
    'iv_fluids': 'iv_fluid_bolus',
    'lactate': 'lactate_measurement',
}
RECOMMENDED_ACTION_TO_DETERIORATING = {
    'emergency_response': 'emergency_response',
    'urgent_review': 'escalate',
    'routine_monitoring': 'monitor',
}
DETERIORATING_ACTION_MAP = {
    'monitor': 'monitor',
    'observe': 'monitor',
    'routine_monitoring': 'monitor',
    'escalate': 'escalate',
    'escalate_care': 'escalate',
    'urgent_review': 'escalate',
    'emergency_response': 'emergency_response',
    'emergency': 'emergency_response',
    'code_blue': 'emergency_response',
}
URGENCY_MAP = {
    'critical': 'immediate',
    'high': 'urgent',
    'medium': 'routine',
    'low': 'routine',
}

# Structured telemetry used by the auto-plot cell after training completes.
REWARD_LOGS = []
REWARD_LOG_STATE = {'event_idx': 0, 'batch_idx': 0}
REWARD_LOG_PATH = os.path.join(OUTPUT_DIR, 'grpo_reward_events.jsonl')
os.makedirs(OUTPUT_DIR, exist_ok=True)

_session = requests.Session()
try:
    _retry = Retry(total=3, backoff_factor=0.7,
                   status_forcelist=[429, 500, 502, 503, 504],
                   allowed_methods=['POST'], raise_on_status=False)
except TypeError:
    _retry = Retry(total=3, backoff_factor=0.7,
                   status_forcelist=[429, 500, 502, 503, 504],
                   method_whitelist=['POST'], raise_on_status=False)
_session.mount('https://', HTTPAdapter(max_retries=_retry))
_session.mount('http://', HTTPAdapter(max_retries=_retry))

def _append_reward_event(event: dict):
    REWARD_LOGS.append(event)
    try:
        with open(REWARD_LOG_PATH, 'a', encoding='utf-8') as f:
            f.write(_json.dumps(event, ensure_ascii=True) + '\n')
    except Exception:
        # Logging should never break training/reward evaluation.
        pass

def _parse_action_dict(completion: str) -> dict:
    txt = re.sub(r'```(?:json)?\s*|\s*```', '', completion or '').strip()
    if not txt:
        return {}
    # Best-effort extraction if model emits wrapper prose around JSON.
    if not txt.startswith('{'):
        m = re.search(r'\{[\s\S]*\}', txt)
        if m:
            txt = m.group(0)
    try:
        parsed = _json.loads(txt)
        # Some outputs come wrapped as {"response": "{...}"}; unwrap once.
        if isinstance(parsed, dict) and isinstance(parsed.get('response'), str):
            inner = parsed['response'].strip()
            if inner.startswith('{') and inner.endswith('}'):
                try:
                    parsed = _json.loads(inner)
                except Exception:
                    pass
        return parsed if isinstance(parsed, dict) else {}
    except Exception:
        return {}

def _normalize_action_dict(task_id: str, action: dict) -> dict:
    if not isinstance(action, dict):
        return {}
    out = dict(action)

    p = out.get('priority')
    if isinstance(p, str):
        out['priority'] = PRIORITY_MAP.get(p.strip().lower(), p)

    if task_id == 'sepsis_bundle':
        elems = out.get('bundle_elements')
        if isinstance(elems, list):
            out['bundle_elements'] = [
                SEPSIS_BUNDLE_TOKEN_MAP.get(str(e).strip(), str(e).strip())
                for e in elems
            ]

    if task_id == 'medication_reconciliation':
        sev = out.get('severity')
        if isinstance(sev, str) and sev.strip().lower() == 'moderate':
            out['severity'] = 'medium'

    if task_id == 'deteriorating_patient':
        act = out.get('action')
        if isinstance(act, str):
            out['action'] = DETERIORATING_ACTION_MAP.get(act.strip().lower(), act.strip().lower())
        if 'action' not in out and isinstance(out.get('recommended_action'), str):
            rec = out['recommended_action'].strip().lower()
            mapped = RECOMMENDED_ACTION_TO_DETERIORATING.get(rec)
            if mapped:
                out['action'] = mapped
        if 'rationale' not in out and isinstance(out.get('assessment'), str):
            out['rationale'] = out['assessment']
        out = {
            'action': out['action'] if out.get('action') in {'monitor', 'escalate', 'emergency_response'} else 'escalate',
            'rationale': out.get('rationale', 'Clinical deterioration risk present.'),
            'confidence': out.get('confidence', 0.5),
        }

    if task_id == 'differential_diagnosis':
        def _norm_text(v):
            s = str(v or '').strip().lower()
            return s.replace(' ', '_').replace('-', '_')

        for k in ('must_not_miss', 'top_diagnosis', 'first_investigation'):
            if k in out and isinstance(out[k], str):
                out[k] = _norm_text(out[k])

        diffs = out.get('differentials')
        if isinstance(diffs, list):
            out['differentials'] = [_norm_text(d) for d in diffs if str(d).strip()]

        urg = out.get('urgency')
        if isinstance(urg, str):
            u = urg.strip().lower()
            out['urgency'] = URGENCY_MAP.get(u, u)

    return out

def reward_fn(completions, prompts=None, task=None, **kwargs):
    batch_size = len(completions)
    task_ids = [task] * batch_size if isinstance(task, str) else (task or ['simple_triage'] * batch_size)
    rewards, success_count = [], 0
    start = time.perf_counter()
    REWARD_LOG_STATE['batch_idx'] += 1
    batch_idx = REWARD_LOG_STATE['batch_idx']

    for idx, (completion, task_id) in enumerate(zip(completions, task_ids)):
        action_dict = _normalize_action_dict(task_id, _parse_action_dict(completion))
        REWARD_LOG_STATE['event_idx'] += 1
        event_idx = REWARD_LOG_STATE['event_idx']

        event = {
            'event_idx': event_idx,
            'batch_idx': batch_idx,
            'task_id': task_id,
            'completion_len': len(completion or ''),
            'action_keys': sorted(action_dict.keys()) if isinstance(action_dict, dict) else [],
            'status': 'ok',
            'reward': FALLBACK_REWARD,
            'is_floor': True,
            'error': '',
            'feedback': '',
            'ts': time.time(),
        }

        try:
            sid = str(uuid.uuid4())
            r1 = _session.post(f'{SERVER_URL}/reset',
                json={'task_id': task_id, 'session_id': sid}, timeout=REQUEST_TIMEOUT)
            r1.raise_for_status()
            r2 = _session.post(f'{SERVER_URL}/step',
                json={'session_id': sid, 'action': action_dict}, timeout=REQUEST_TIMEOUT)
            r2.raise_for_status()
            step_json = r2.json()
            reward = float(step_json['reward'])
            success_count += 1

            info = step_json.get('info', {}) if isinstance(step_json, dict) else {}
            feedback = info.get('feedback', '') if isinstance(info, dict) else ''
            event['reward'] = reward
            event['is_floor'] = reward <= FALLBACK_REWARD
            event['feedback'] = str(feedback)[:240]

            if DEBUG_LOGGING:
                print(f'  [{task_id}] reward={reward:.4f} ({idx+1}/{batch_size})')
                if reward <= FALLBACK_REWARD:
                    keys = sorted(action_dict.keys()) if isinstance(action_dict, dict) else []
                    print(f'    floor-dbg keys={keys} feedback={event["feedback"] or "n/a"}')
        except Exception as e:
            reward = FALLBACK_REWARD
            event['status'] = 'error'
            event['error'] = str(e)[:240]
            if DEBUG_LOGGING:
                print(f'  [error] {task_id}: {str(e)[:60]}')

        _append_reward_event(event)
        rewards.append(reward)

    elapsed = time.perf_counter() - start
    print(f'\nBatch: {success_count}/{batch_size} OK | {elapsed:.2f}s\n')
    return rewards

# Smoke test using paired session (same case for generate + grade)
print('Running smoke test...')
_sid = str(uuid.uuid4())
_r1 = requests.post(f'{SERVER_URL}/reset', json={'task_id': 'simple_triage', 'session_id': _sid}, timeout=30)
_patient = _r1.json()['observation'].get('patient_history', '')
print('Case:', _patient[:100])
_action = {'priority': 'high', 'news2_score': 7, 'critical_sign': 'spo2',
           'recommended_action': 'emergency_response', 'rationale': 'test', 'confidence': 0.9}
_r2 = requests.post(f'{SERVER_URL}/step',
    json={'session_id': _sid, 'action': _action}, timeout=30)
print('Smoke test reward:', _r2.json()['reward'])
print('(Any value > 0.0001 confirms the API pipeline is working correctly)')
print(f'Reward events will stream to: {REWARD_LOG_PATH}')

In [ ]:
# Cell 10 — GRPO training (TRL 1.1.0 compatible + auto metrics report)
import os
import json
import trl
import matplotlib.pyplot as plt
from collections import defaultdict, deque
print(f"TRL Version: {trl.__version__}")
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=NUM_GENERATIONS,
    # 260 keeps harder-task JSON from getting cut off.
    max_completion_length=260,
    learning_rate=1e-5,
    lr_scheduler_type='cosine',
    logging_steps=1,
    save_steps=10,
    save_total_limit=3,
    fp16=False,
    bf16=True,
    report_to='none',
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=config,
    train_dataset=dataset,
    reward_funcs=[reward_fn],
)

def quick_preflight_hard_tasks(samples_per_task=1):
    """Generate a tiny sample on hard tasks and score via reward_fn before full train."""
    hard_tasks = ['differential_diagnosis', 'deteriorating_patient', 'masked_deterioration']
    passed = 0
    total = 0

    task_column = dataset['task']
    print('\nRunning hard-task preflight...')

    for task_id in hard_tasks:
        idxs = [i for i, t in enumerate(task_column) if t == task_id][:samples_per_task]
        if not idxs:
            continue
        for i in idxs:
            prompt_text = dataset[i]['prompt']
            inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=220,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            completion = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            reward = reward_fn([completion], task=[task_id])[0]
            ok = reward > 0.0001
            passed += int(ok)
            total += 1
            print(f'  preflight {task_id}: reward={reward:.4f} | len={len(completion)}')

    print(f'Preflight result: {passed}/{total} above floor')
    if total > 0 and passed == 0:
        print('WARNING: hard-task preflight is all-floor; do not start long run yet.')

def _latest_checkpoint_path(output_dir: str):
    import re
    if not os.path.isdir(output_dir):
        return None
    names = [d for d in os.listdir(output_dir) if d.startswith('checkpoint-')]
    pairs = []
    for n in names:
        m = re.search(r'checkpoint-(\d+)$', n)
        if m:
            pairs.append((int(m.group(1)), n))
    if not pairs:
        return None
    pairs.sort(key=lambda x: x[0])
    return os.path.join(output_dir, pairs[-1][1])

def _rolling(values, window):
    out = []
    q = deque()
    s = 0.0
    for v in values:
        q.append(v)
        s += v
        if len(q) > window:
            s -= q.popleft()
        out.append(s / len(q))
    return out

def _load_reward_events(path: str):
    if 'REWARD_LOGS' in globals() and REWARD_LOGS:
        return REWARD_LOGS
    rows = []
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rows.append(json.loads(line))
                except Exception:
                    continue
    return rows

def render_training_report(window=20):
    events = _load_reward_events(REWARD_LOG_PATH)
    if not events:
        print('No reward events found yet; skipping report.')
        return

    events = sorted(events, key=lambda x: x.get('event_idx', 0))
    xs = list(range(1, len(events) + 1))
    rewards = [float(e.get('reward', 0.0001)) for e in events]
    floors = [1 if float(e.get('reward', 0.0001)) <= 0.0001 else 0 for e in events]
    errors = [1 if e.get('status') == 'error' else 0 for e in events]

    by_task = defaultdict(list)
    for i, e in enumerate(events, start=1):
        by_task[str(e.get('task_id', 'unknown'))].append((i, float(e.get('reward', 0.0001))))

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    axes[0, 0].plot(xs, rewards, alpha=0.25, label='raw reward')
    axes[0, 0].plot(xs, _rolling(rewards, window), linewidth=2, label=f'rolling-{window}')
    axes[0, 0].set_title('Overall Reward Trend')
    axes[0, 0].set_xlabel('Reward event')
    axes[0, 0].set_ylabel('Reward')
    axes[0, 0].set_ylim(0, 1.02)
    axes[0, 0].legend()

    for task_name in sorted(by_task.keys()):
        task_x = [p[0] for p in by_task[task_name]]
        task_y = [p[1] for p in by_task[task_name]]
        axes[0, 1].plot(task_x, _rolling(task_y, max(5, window // 2)), label=task_name)
    axes[0, 1].set_title('Task-wise Rolling Reward')
    axes[0, 1].set_xlabel('Reward event')
    axes[0, 1].set_ylabel('Reward')
    axes[0, 1].set_ylim(0, 1.02)
    axes[0, 1].legend(fontsize=8, ncol=2)

    axes[1, 0].plot(xs, _rolling(floors, window), color='tab:red', linewidth=2)
    axes[1, 0].set_title('Floor-rate Trend (reward <= 0.0001)')
    axes[1, 0].set_xlabel('Reward event')
    axes[1, 0].set_ylabel('Floor rate')
    axes[1, 0].set_ylim(0, 1.02)

    axes[1, 1].plot(xs, _rolling(errors, window), color='tab:orange', linewidth=2)
    axes[1, 1].set_title('Error-rate Trend (request/422/etc.)')
    axes[1, 1].set_xlabel('Reward event')
    axes[1, 1].set_ylabel('Error rate')
    axes[1, 1].set_ylim(0, 1.02)

    plt.tight_layout()
    plt.show()

    task_summary = []
    for task_name in sorted(by_task.keys()):
        vals = [y for _, y in by_task[task_name]]
        floor_count = sum(1 for v in vals if v <= 0.0001)
        task_summary.append({
            'task': task_name,
            'n': len(vals),
            'mean_reward': round(sum(vals) / len(vals), 4),
            'floor_rate': round(floor_count / len(vals), 4),
        })

    print('\n=== Reward Summary by Task ===')
    print('task                         n   mean_reward   floor_rate')
    for row in task_summary:
        print(f"{row['task']:<28} {row['n']:>3}     {row['mean_reward']:.4f}       {row['floor_rate']:.4f}")

    print('\nOverall:')
    print(f"  events={len(events)} | mean_reward={sum(rewards)/len(rewards):.4f} | floor_rate={sum(floors)/len(floors):.4f} | error_rate={sum(errors)/len(errors):.4f}")
    print(f'  reward log file: {REWARD_LOG_PATH}')

# ---------------- Resume / fresh run control ----------------
# WHEN TO USE:
# - RUN_MODE='fresh'         -> first full run from step 0
# - RUN_MODE='resume_latest' -> Colab disconnected / aborted; continue from latest checkpoint-*
# - RUN_MODE='resume_path'   -> continue from a specific checkpoint path
RUN_MODE = 'fresh'
RESUME_CHECKPOINT_PATH = ''  # used only when RUN_MODE='resume_path'

print(f'Training: {len(dataset)} prompts x {EPOCHS} epoch(s), G={NUM_GENERATIONS}')
print(f'Checkpoints saving to: {OUTPUT_DIR} every 10 steps')
print(f'Reward telemetry file: {REWARD_LOG_PATH}')

if RUN_MODE == 'fresh':
    # Fresh run: run preflight first to catch obvious schema issues early.
    quick_preflight_hard_tasks(samples_per_task=1)
    trainer.train()
elif RUN_MODE == 'resume_latest':
    ckpt = _latest_checkpoint_path(OUTPUT_DIR)
    if not ckpt:
        raise RuntimeError(f'No checkpoint-* found under {OUTPUT_DIR}; use RUN_MODE=\'fresh\' first.')
    print(f'Resuming from latest checkpoint: {ckpt}')
    trainer.train(resume_from_checkpoint=ckpt)
elif RUN_MODE == 'resume_path':
    if not RESUME_CHECKPOINT_PATH:
        raise RuntimeError('Set RESUME_CHECKPOINT_PATH when RUN_MODE=\'resume_path\'.')
    print(f'Resuming from explicit checkpoint: {RESUME_CHECKPOINT_PATH}')
    trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT_PATH)
else:
    raise ValueError("RUN_MODE must be 'fresh', 'resume_latest', or 'resume_path'.")

# Auto-build charts immediately after run completion.
render_training_report(window=20)

In [ ]:
# Cell 11 — Save final adapter to Google Drive
import os
save_path = os.path.join(OUTPUT_DIR, 'final')
os.makedirs(save_path, exist_ok=True)
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f'Adapter saved to: {save_path}')
print('Files:', os.listdir(save_path))

In [ ]:
# Cell 12 — (Optional) Push adapter to HF Hub
hub_repo_id = 'kunalkachru23/grpo-medical-triage-qwen1.5b'

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    model.push_to_hub(hub_repo_id)
    tokenizer.push_to_hub(hub_repo_id)
    print(f'Pushed to: https://huggingface.co/{hub_repo_id}')
else:
    print('HF_TOKEN not set — skipping. Set HF_TOKEN in Cell 3 to enable.')

In [ ]:
# Cell 13 — Recovery loader for historical runs (telemetry JSONL or trainer_state fallback)
import os, json, glob

FALLBACK_REWARD = 0.0001
OUTPUT_DIR = globals().get('OUTPUT_DIR', os.path.abspath('grpo-medical-triage'))
REWARD_LOG_PATH = globals().get('REWARD_LOG_PATH', os.path.join(OUTPUT_DIR, 'grpo_reward_events.jsonl'))

def _read_jsonl(path):
    rows = []
    if path and os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rows.append(json.loads(line))
                except Exception:
                    pass
    return rows

def _best_file(patterns):
    files = []
    for p in patterns:
        files.extend(glob.glob(p, recursive=True))
    files = sorted(set(files))
    if not files:
        return ''
    files.sort(key=lambda p: os.path.getsize(p), reverse=True)
    return files[0]

def _reconstruct_from_trainer_state(path):
    rows = []
    if not path or not os.path.exists(path):
        return rows
    with open(path, 'r', encoding='utf-8') as f:
        state = json.load(f)
    hist = state.get('log_history', [])
    idx = 0
    for rec in hist:
        reward = None
        for k in ('reward', 'rewards/mean', 'mean_reward', 'objective/reward', 'train/reward'):
            if k in rec and isinstance(rec[k], (int, float)):
                reward = float(rec[k])
                break
        if reward is None:
            continue
        idx += 1
        rows.append({
            'event_idx': idx,
            'task_id': 'unknown',
            'reward': reward,
            'status': 'ok',
            'is_floor': reward <= FALLBACK_REWARD,
            'error': '',
            'step': rec.get('step'),
            'epoch': rec.get('epoch'),
        })
    return rows

REWARD_LOGS = []
source = 'none'
if 'REWARD_LOGS' in globals() and isinstance(globals()['REWARD_LOGS'], list) and len(globals()['REWARD_LOGS']) > 0:
    REWARD_LOGS = globals()['REWARD_LOGS']
    source = 'runtime memory'

if not REWARD_LOGS:
    REWARD_LOGS = _read_jsonl(REWARD_LOG_PATH)
    if REWARD_LOGS:
        source = f'telemetry jsonl ({REWARD_LOG_PATH})'

if not REWARD_LOGS:
    best_jsonl = _best_file([
        os.path.join(OUTPUT_DIR, '**', 'grpo_reward_events.jsonl'),
        os.path.join(os.getcwd(), '**', 'grpo_reward_events.jsonl'),
    ])
    if best_jsonl:
        REWARD_LOG_PATH = best_jsonl
        REWARD_LOGS = _read_jsonl(best_jsonl)
        if REWARD_LOGS:
            source = f'auto-found telemetry jsonl ({best_jsonl})'

if not REWARD_LOGS:
    best_state = _best_file([
        os.path.join(OUTPUT_DIR, '**', 'trainer_state.json'),
        os.path.join(os.getcwd(), '**', 'trainer_state.json'),
    ])
    REWARD_LOGS = _reconstruct_from_trainer_state(best_state)
    if REWARD_LOGS:
        source = f'reconstructed from trainer_state ({best_state})'

events = REWARD_LOGS
print(f'Loaded events: {len(events)}')
print(f'Source: {source}')
if not events:
    raise RuntimeError('No reward events found. Re-run Cell 9 + Cell 10 once, or paste raw logs for parsing.')
print('Sample keys:', sorted(events[0].keys()))

In [ ]:
# Cell 14 — Recovery charts (clean visuals, reconstructed-run safe)
import matplotlib.pyplot as plt
from collections import defaultdict, deque

FALLBACK_REWARD = 0.0001
events = sorted(events, key=lambda x: x.get('event_idx', 0))

def rolling(vals, window=20):
    out, q, s = [], deque(), 0.0
    for v in vals:
        q.append(v)
        s += v
        if len(q) > window:
            s -= q.popleft()
        out.append(s / len(q))
    return out

xs = list(range(1, len(events) + 1))
rewards = [float(e.get('reward', FALLBACK_REWARD)) for e in events]
floors = [1 if r <= FALLBACK_REWARD else 0 for r in rewards]
errors = [1 if str(e.get('status', 'ok')).lower() == 'error' else 0 for e in events]

by_task = defaultdict(list)
for i, e in enumerate(events, start=1):
    t = str(e.get('task_id', 'unknown'))
    by_task[t].append((i, float(e.get('reward', FALLBACK_REWARD))))

has_real_tasks = any(t != 'unknown' for t in by_task.keys())
W = 20

plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(2, 2, figsize=(16, 10))

ax[0, 0].plot(xs, rewards, color='#4C78A8', alpha=0.25, linewidth=1.3, label='raw reward')
ax[0, 0].plot(xs, rolling(rewards, W), color='#1F4E79', linewidth=2.6, label=f'rolling mean ({W})')
ax[0, 0].axhline(FALLBACK_REWARD, linestyle='--', linewidth=1, color='crimson', alpha=0.7, label='floor')
ax[0, 0].set_title('Overall Reward Trend', fontsize=12, weight='bold')
ax[0, 0].set_xlabel('Training event')
ax[0, 0].set_ylabel('Reward')
ax[0, 0].set_ylim(0, 1.02)
ax[0, 0].legend(loc='lower right')

if has_real_tasks:
    for task in sorted(by_task):
        tx = [p[0] for p in by_task[task]]
        ty = [p[1] for p in by_task[task]]
        ax[0, 1].plot(tx, rolling(ty, max(5, W // 2)), linewidth=1.8, label=task)
    ax[0, 1].set_title('Task-wise Rolling Reward', fontsize=12, weight='bold')
    ax[0, 1].legend(fontsize=8, ncol=2)
else:
    ax[0, 1].plot(xs, rolling(rewards, W), color='#2A9D8F', linewidth=2.8, label='rolling reward')
    ax[0, 1].fill_between(xs, [0] * len(xs), rolling(rewards, W), color='#2A9D8F', alpha=0.12)
    ax[0, 1].set_title('Task-wise View Unavailable (Reconstructed Run)', fontsize=12, weight='bold')
    ax[0, 1].text(
        0.5,
        0.5,
        'Historical run reconstructed from trainer_state.json\nPer-task IDs not available in this log source.',
        transform=ax[0, 1].transAxes,
        ha='center',
        va='center',
        fontsize=10,
        color='#444',
    )

ax[0, 1].set_xlabel('Training event')
ax[0, 1].set_ylabel('Reward')
ax[0, 1].set_ylim(0, 1.02)

ax[1, 0].plot(xs, rolling(floors, W), color='#D62728', linewidth=2.6)
ax[1, 0].set_title('Floor-rate Trend (reward <= 0.0001)', fontsize=12, weight='bold')
ax[1, 0].set_xlabel('Training event')
ax[1, 0].set_ylabel('Rate')
ax[1, 0].set_ylim(0, 1.02)

ax[1, 1].plot(xs, rolling(errors, W), color='#F58518', linewidth=2.6)
ax[1, 1].set_title('Error-rate Trend', fontsize=12, weight='bold')
ax[1, 1].set_xlabel('Training event')
ax[1, 1].set_ylabel('Rate')
ax[1, 1].set_ylim(0, 1.02)

for row in ax:
    for a in row:
        a.grid(alpha=0.25)
        a.spines['top'].set_visible(False)
        a.spines['right'].set_visible(False)

overall_mean = sum(rewards) / len(rewards)
overall_floor = sum(floors) / len(floors)
overall_err = sum(errors) / len(errors)
fig.suptitle(
    f'GRPO Training Diagnostics | events={len(events)} | mean={overall_mean:.4f} | floor={overall_floor:.4f} | error={overall_err:.4f}',
    fontsize=13,
    weight='bold',
    y=1.02,
)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 15 — Save clean report artifacts + markdown snippet for docs/EVIDENCE_SUMMARY.md
import os, json, csv, time
from pathlib import Path
from collections import defaultdict, deque
import matplotlib.pyplot as plt

FALLBACK_REWARD = 0.0001
events = sorted(events, key=lambda x: x.get('event_idx', 0))

def rolling(vals, window=20):
    out, q, s = [], deque(), 0.0
    for v in vals:
        q.append(v)
        s += v
        if len(q) > window:
            s -= q.popleft()
        out.append(s / len(q))
    return out

xs = list(range(1, len(events) + 1))
rewards = [float(e.get('reward', FALLBACK_REWARD)) for e in events]
floors = [1 if r <= FALLBACK_REWARD else 0 for r in rewards]
errors = [1 if str(e.get('status', 'ok')).lower() == 'error' else 0 for e in events]

by_task = defaultdict(list)
task_err = defaultdict(int)
for i, e in enumerate(events, start=1):
    t = str(e.get('task_id', 'unknown'))
    r = float(e.get('reward', FALLBACK_REWARD))
    by_task[t].append((i, r))
    if str(e.get('status', 'ok')).lower() == 'error':
        task_err[t] += 1

has_real_tasks = any(t != 'unknown' for t in by_task.keys())

OUTPUT_DIR = globals().get('OUTPUT_DIR', os.path.abspath('grpo-medical-triage'))
ARTIFACT_DIR = Path(OUTPUT_DIR) / 'training_artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
stamp = time.strftime('%Y%m%d-%H%M%S')

png_path = ARTIFACT_DIR / f'training_report_clean_{stamp}.png'
csv_path = ARTIFACT_DIR / f'task_summary_{stamp}.csv'
json_path = ARTIFACT_DIR / f'metrics_{stamp}.json'

W = 20
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(2, 2, figsize=(16, 10))

ax[0, 0].plot(xs, rewards, color='#4C78A8', alpha=0.25, linewidth=1.3, label='raw reward')
ax[0, 0].plot(xs, rolling(rewards, W), color='#1F4E79', linewidth=2.6, label=f'rolling mean ({W})')
ax[0, 0].axhline(FALLBACK_REWARD, linestyle='--', linewidth=1, color='crimson', alpha=0.7, label='floor')
ax[0, 0].set_title('Overall Reward Trend', fontsize=12, weight='bold')
ax[0, 0].set_ylim(0, 1.02)
ax[0, 0].legend(loc='lower right')

if has_real_tasks:
    for t in sorted(by_task):
        tx = [p[0] for p in by_task[t]]
        ty = [p[1] for p in by_task[t]]
        ax[0, 1].plot(tx, rolling(ty, max(5, W // 2)), linewidth=1.8, label=t)
    ax[0, 1].set_title('Task-wise Rolling Reward', fontsize=12, weight='bold')
    ax[0, 1].legend(fontsize=8, ncol=2)
else:
    ax[0, 1].plot(xs, rolling(rewards, W), color='#2A9D8F', linewidth=2.8)
    ax[0, 1].fill_between(xs, [0] * len(xs), rolling(rewards, W), color='#2A9D8F', alpha=0.12)
    ax[0, 1].set_title('Task-wise View Unavailable (Reconstructed Run)', fontsize=12, weight='bold')
    ax[0, 1].text(
        0.5,
        0.5,
        'Historical run reconstructed from trainer_state.json\nPer-task IDs not available in this log source.',
        transform=ax[0, 1].transAxes,
        ha='center',
        va='center',
        fontsize=10,
        color='#444',
    )

ax[0, 1].set_ylim(0, 1.02)
ax[1, 0].plot(xs, rolling(floors, W), color='#D62728', linewidth=2.6)
ax[1, 0].set_title('Floor-rate Trend (reward <= 0.0001)', fontsize=12, weight='bold')
ax[1, 0].set_ylim(0, 1.02)
ax[1, 1].plot(xs, rolling(errors, W), color='#F58518', linewidth=2.6)
ax[1, 1].set_title('Error-rate Trend', fontsize=12, weight='bold')
ax[1, 1].set_ylim(0, 1.02)

for row in ax:
    for a in row:
        a.grid(alpha=0.25)
        a.spines['top'].set_visible(False)
        a.spines['right'].set_visible(False)

fig.suptitle(
    f'GRPO Training Diagnostics | events={len(events)} | mean={sum(rewards)/len(rewards):.4f} | floor={sum(floors)/len(floors):.4f} | error={sum(errors)/len(errors):.4f}',
    fontsize=13,
    weight='bold',
    y=1.02,
)
plt.tight_layout()
fig.savefig(png_path, dpi=180, bbox_inches='tight')
plt.close(fig)

rows = []
for t in sorted(by_task):
    vals = [v for _, v in by_task[t]]
    n = len(vals)
    floor_n = sum(1 for v in vals if v <= FALLBACK_REWARD)
    rows.append({
        'task': t,
        'n': n,
        'mean_reward': round(sum(vals) / n, 6),
        'floor_rate': round(floor_n / n, 6),
        'error_rate': round(task_err[t] / n, 6),
    })

with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['task', 'n', 'mean_reward', 'floor_rate', 'error_rate'])
    w.writeheader()
    w.writerows(rows)

overall = {
    'events': len(events),
    'mean_reward': sum(rewards) / len(rewards),
    'floor_rate': sum(floors) / len(floors),
    'error_rate': sum(errors) / len(errors),
    'source': 'reconstructed_from_trainer_state' if not has_real_tasks else 'telemetry_jsonl_or_runtime',
    'taskwise_available': has_real_tasks,
}
payload = {
    'generated_at': stamp,
    'overall': overall,
    'task_summary': rows,
    'artifacts': {
        'plot_png': str(png_path),
        'task_csv': str(csv_path),
        'metrics_json': str(json_path),
    },
}
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2)

print('Saved artifacts:')
print(' -', png_path)
print(' -', csv_path)
print(' -', json_path)

md = f"""## Training Evidence (Auto-generated)\n\n- Reward events: {overall['events']}\n- Mean reward: {overall['mean_reward']:.4f}\n- Floor rate (`<= 0.0001`): {overall['floor_rate']:.4f}\n- Error rate: {overall['error_rate']:.4f}\n- Source: `{overall['source']}`\n\n### Notes\n- This run was reconstructed from `trainer_state.json`; task-wise breakdown is limited.\n\n### Artifacts\n- Plot: `{png_path}`\n- Task summary CSV: `{csv_path}`\n- Metrics JSON: `{json_path}`\n"""
print('\nPaste into docs/EVIDENCE_SUMMARY.md:\n')
print(md)